In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'data_source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sports-consolidated-dp/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed'

# DEFINE THE TABLES
bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.{data_source}'

# Base Path:  s3://sportsbar-final/orders
# Landing Path:  s3://sportsbar-final/orders/landing/
# Processed Path:  s3://sportsbar-final/orders/processed/

In [ ]:
# ---> BRONZE <---
df = spark.read.options(header=True, inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_timestamp', F.current_timestamp()).select("*", '_metadata.file_name', '_metadata.file_size')

# 51810
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 5 rows

In [ ]:
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('append') \
    .saveAsTable(bronze_table)

In [ ]:
# ---> MOVING FILES FROM SOURCE TO PROCESSED DIRECTORY WITHIN S3 BUCKET <---
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f'{processed_path}/{file_info.name}',
        True
    )

In [ ]:
# ---> SILVER <---
df_orders = spark.sql(f'SELECT * FROM {bronze_table}')
df_orders.show(2)

# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2025-11-24 13:32:...|orders_2025_07_01...|    20744|
# |FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2025-11-24 13:32:...|orders_2025_07_01...|    20744|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 2 rows

In [ ]:
# Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col('order_qty')).isNotNull()

# Clean customer_id -< keep numeric otherwise set to 999999
df_orders = df_orders.withColumn(
    'customer_id',
    F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
     .otherwise('999999')
     .cast('string')
)

# Remove weekday name from the date text
# 'Tuesday, July 01, 2025' -> 'July 01, 2025'
df_orders = df_orders.withColumn(
    'orders_replacement_date',
    F.regexp_replace(F.col('order_replacement_date', r'^[A-Za-z]+,\s*', ''))
)

# Parse order_replacement_date using multiple possible formats
df_orders = df_orders.withColumn(
    'orders_replacement_date',
    F.coalesce(
        F.try_to_date('order_replacement_date', 'yyyy/MM/dd'),
        F.try_to_date('order_replacement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_replacement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_replacement_date', 'MMMM dd, yyyy')
    )
)

# Drop duplicates
df_orders = df_orders.dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])

# Check what's the maximum and minimum date
df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).show()